<center>
    <h1>1.0.0 Mention-level Contextualized Embeddings</h1>
    <hr style="width: 70%; margin: 1.5em auto; border: none; border-top: 1.5px solid #fcf8e6;">
</center>


This notebook generates mention-level contextualized embeddings using SciBERT. Each mention is represented by the mean of the token vectors corresponding to the mention span, extracted from a centered context window. The resulting embeddings are saved per-bin for downstream analysis of semantic change.


<center>
<hr style="width: 70%; margin: 1.5em auto; border: none; border-top: 1.5px solid #fcf8e6;">
    <h2>Purpose</h2>
</center>

This notebook converts the accepted analysis corpus bin files into contextualized mention embeddings with SciBERT. Each saved vector represents a `"dark matter"` mention in its local sentence-centered context, and the notebook writes one embedding tensor per time bin for each selected layer-pooling mode.

<center>
<hr style="width: 70%; margin: 1.5em auto; border: none; border-top: 1.5px solid #fcf8e6;">
    <h2>Main Variables To Set</h2>
</center>

- `RUN_ID`: which corpus run to embed
- `MODEL_PATH`: local SciBERT checkpoint to use
- `MAX_LENGTH`, `LEFT_CHARS`, `RIGHT_CHARS`: context-window and truncation behavior
- `LAYER_MODES`: which hidden-state aggregation strategies to compute
- any cache / manifest checks that control when existing embedding files should be reused versus recomputed

<center>
<hr style="width: 70%; margin: 1.5em auto; border: none; border-top: 1.5px solid #fcf8e6;">
    <h2>Outputs</h2>
</center>

Embeddings are written to `outputs/analytical-results / RUN_ID / embeddings / <layer_mode> /`, together with per-bin metadata JSON files recording row counts, failure counts, and the corpus hash they were built from. Downstream notebooks expect these bin-wise tensors to stay row-aligned with the corpus index.


<hr style="width: 95%; margin: 1.5em auto; border: none; border-top: 1.5px solid #fcf8e6;">


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path


def _find_workspace_root() -> Path:
    env_root = os.environ.get("WORKSPACE_ROOT", "").strip()
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if (candidate / "config" / "workspace.json").exists():
            return candidate
    for start in [Path.cwd(), *Path.cwd().parents]:
        if (start / "config" / "workspace.json").exists():
            return start
    raise FileNotFoundError(
        "Could not locate workspace root from WORKSPACE_ROOT or the current working directory."
    )


WORKSPACE_DIR = _find_workspace_root()
PAPER_ROOT = WORKSPACE_DIR.parent
RESEARCH_ROOT = PAPER_ROOT.parent
SHARED_ASSETS_DIR = RESEARCH_ROOT / "shared-assets"
SHARED_CODE_DIR = SHARED_ASSETS_DIR / "code"
if str(SHARED_CODE_DIR) not in sys.path:
    sys.path.insert(0, str(SHARED_CODE_DIR))

CODE_DIR = WORKSPACE_DIR / "code"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from workspace_rooting.workspace_paths import canonical_workspace_paths

paths = canonical_workspace_paths(WORKSPACE_DIR)
WORKSPACE_DIR = paths["workspace"]
CODE_DIR = paths["code"]
CONFIG_DIR = paths["config"]
DATA_DIR = paths["data"]
OUTPUTS_DIR = paths["outputs"]
DOCS_DIR = paths["docs"]
LOCAL_DIR = paths["local"]
SHARED_ASSETS_DIR = paths["shared_assets"]

print("workspace_root:", WORKSPACE_DIR)
print("shared_assets_root:", SHARED_ASSETS_DIR)

import json, time, sys
from pathlib import Path
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
import torch

tqdm.pandas()

RUN_ID = "analysis_corpus"

corpus_root = LOCAL_DIR / "corpora" / RUN_ID
CORPUS_DIR = corpus_root / "bins"
corpus_manifest_path = corpus_root / "corpus_manifest.json"

run_root = OUTPUTS_DIR / "analytical-results" / RUN_ID
EMB_ROOT = run_root / "embeddings"

print("CORPUS_DIR:", CORPUS_DIR, "exists:", CORPUS_DIR.exists())
print("EMB_ROOT:", EMB_ROOT, "exists:", EMB_ROOT.exists())
print("corpus_manifest_path:", corpus_manifest_path, "exists:", corpus_manifest_path.exists())

EMB_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_PATH = str(LOCAL_DIR / "models" / "scibert_scivocab_uncased")
DEVICE = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")

MAX_LENGTH = 512
BATCH_SIZE = 16
LEFT_CHARS = 550
RIGHT_CHARS = 550

POOL = "mention_mean"
LAYER_CONFIGS = {
    "last1": lambda hs: hs[-1],
    "last4_mean": lambda hs: torch.stack(hs[-4:], dim=0).mean(dim=0),
    "mid4_mean": lambda hs: torch.stack(hs[4:8], dim=0).mean(dim=0),
    "all_mean": lambda hs: torch.stack(hs[1:], dim=0).mean(dim=0),
    "weighted": None,
}
AVAILABLE_LAYER_MODES = ["last1", "last4_mean", "mid4_mean", "all_mean", "weighted"]
BEST_LAYER_MODE = "all_mean"
LAYER_MODE_POLICY = "best_only"  # set to "all" to unrestrict the run
EXPLICIT_LAYER_MODES = None  # e.g. ["all_mean"] to run a custom subset explicitly


def resolve_layer_modes():
    if EXPLICIT_LAYER_MODES is not None:
        requested = list(EXPLICIT_LAYER_MODES)
    elif LAYER_MODE_POLICY == "best_only":
        requested = [BEST_LAYER_MODE]
    elif LAYER_MODE_POLICY == "all":
        requested = list(AVAILABLE_LAYER_MODES)
    else:
        raise ValueError(
            f"Unsupported LAYER_MODE_POLICY={LAYER_MODE_POLICY!r}. Use 'all' or 'best_only'."
        )

    unknown = [mode for mode in requested if mode not in LAYER_CONFIGS]
    if unknown:
        raise ValueError(f"Unknown layer mode(s): {unknown}")
    return requested


LAYER_MODES = resolve_layer_modes()
print("Layer modes to run:", LAYER_MODES)


<center>
<hr style="width: 70%; margin: 0.5em auto; border: none; border-top: 1.5px solid #fcf8e6;">
    <h2>Definitions, functions, helpers</h2>
</center>


In [ ]:
def log(msg: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')}] {msg}")
    sys.stdout.flush()

def bin_start_year(label: str) -> int:
    return int(label.split("-")[0])

def load_jsonl_records(path: Path):
    recs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            recs.append(json.loads(line))
    return recs

def attention_weighted_pool(h_i, attentions, mention_tok, batch_idx):
    # h_i: (1, T, d) — single item
    # attentions: tuple of (bsz, heads, T, T) per layer
    attn = torch.stack(attentions[-4:], dim=0).mean(dim=0).mean(dim=1)  # (bsz, T, T)
    attn_i = attn[batch_idx]          # (T, T) — single item
    mention_attn = attn_i[mention_tok, :].mean(dim=0, keepdim=True)  # (1, T)
    mention_attn = mention_attn / mention_attn.sum(dim=-1, keepdim=True).clamp_min(1e-10)
    return torch.bmm(mention_attn.unsqueeze(0), h_i).squeeze(0).squeeze(0)  # (d,)

def center_window(text: str, start: int, end: int, left_chars: int, right_chars: int):
    """
    Return (window_text, new_start, new_end) such that the mention span remains
    inside the window and offsets are adjusted.
    """
    L = len(text)
    win_left = max(0, start - left_chars)
    win_right = min(L, end + right_chars)

    # snap to whitespace boundaries to avoid chopping tokens mid-word
    while win_left > 0 and not text[win_left - 1].isspace():
        win_left -= 1
    while win_right < L and not text[win_right].isspace():
        win_right += 1

    w = text[win_left:win_right]
    return w, start - win_left, end - win_left

def _span_token_indices(offsets, start_char: int, end_char: int):
    idx = []
    for ti, (a, b) in enumerate(offsets):
        if a == b == 0:
            continue
        if b > start_char and a < end_char:
            idx.append(ti)
    return idx

@torch.inference_mode()
def embed_mentions_centered(records, tokenizer, model, layer_mode="last4_mean",
                             left_chars=550, right_chars=550):
    model.eval()

    texts, starts, ends = [], [], []
    for r in records:
        w, ns, ne = center_window(
            r["sent"], int(r["start"]), int(r["end"]),
            left_chars=left_chars, right_chars=right_chars
        )
        texts.append(w)
        starts.append(ns)
        ends.append(ne)

    n_truncated = sum(
        1 for t in texts
        if len(tokenizer(t, truncation=False)["input_ids"]) >= MAX_LENGTH
    )
    if n_truncated > 0:
        print(f"  Truncated: {n_truncated}/{len(texts)} ({n_truncated/len(texts):.1%})")

    all_vecs = []
    n = len(records)
    batch_size = 8 if layer_mode == "weighted" else 16
    n_batches = (n + batch_size - 1) // batch_size
    # and use batch_size everywhere inside the function instead of BATCH_SIZE
    n_failed = 0

    for bi in range(n_batches):
        s0 = bi * batch_size
        s1 = min(n, s0 + batch_size)

        batch_texts = texts[s0:s1]
        batch_starts = starts[s0:s1]
        batch_ends = ends[s0:s1]

        tok = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
            return_offsets_mapping=True,
        )

        offsets = tok.pop("offset_mapping")  # (b,t,2)
        offsets = offsets.to("cpu").numpy()
        tok = {k: v.to(DEVICE) for k, v in tok.items()}

        out = model(**tok, output_hidden_states=True,
                    output_attentions=(layer_mode == "weighted"))
        hs = out.hidden_states  # tuple of (bsz, T, d) per layer

        if layer_mode == "last1":
            h = hs[-1]
        elif layer_mode == "last4_mean":
            h = torch.stack(hs[-4:], dim=0).mean(dim=0)
        elif layer_mode == "mid4_mean":
            h = torch.stack(list(hs[6:10]), dim=0).mean(dim=0)
        elif layer_mode == "all_mean":
            h = torch.stack(list(hs[1:]), dim=0).mean(dim=0)
        elif layer_mode == "weighted":
            h = hs[-1]  # base for weighted — attentions handled per-item below


        bsz, T, d = h.shape
        vecs = torch.empty((bsz, d), device=h.device, dtype=h.dtype)

        for i in range(bsz):
            mention_tok = _span_token_indices(offsets[i], batch_starts[i], batch_ends[i])

            if not mention_tok:
                n_failed += 1
                # fallback: mean over non-pad tokens
                attn = tok["attention_mask"][i].unsqueeze(-1)
                v = (h[i] * attn).sum(dim=0) / attn.sum().clamp_min(1)
                vecs[i] = v
                continue

            if layer_mode == "weighted":
                v = attention_weighted_pool(
                    h[i].unsqueeze(0),
                    out.attentions,
                    mention_tok,
                    batch_idx=i
                )
            else:
                v = h[i, mention_tok, :].mean(dim=0)
            vecs[i] = v

        vecs = torch.nn.functional.normalize(vecs, p=2, dim=1)
        all_vecs.append(vecs.detach().to("cpu", dtype=torch.float32))

    X = torch.cat(all_vecs, dim=0)
    return X, n_failed


<center>
<hr style="width: 70%; margin: 0.5em auto; border: none; border-top: 1.5px solid #fcf8e6;">
    <h2>Main loop execution</h2>
</center>


In [ ]:
# ── Safe-guard: manifest + bins ───────────────────────────────────────────
if corpus_manifest_path.exists():
    with open(corpus_manifest_path, "r", encoding="utf-8") as f:
        corpus_manifest = json.load(f)

    bins = corpus_manifest.get("written_bins", [])
    if not bins:
        bins = sorted([p.stem for p in CORPUS_DIR.glob("*.jsonl")], key=bin_start_year)
        print("Manifest found, but no written_bins listed; falling back to directory scan.")
    else:
        print(f"Loaded {len(bins)} bins from corpus manifest.")
else:
    corpus_manifest = None
    bins = sorted([p.stem for p in CORPUS_DIR.glob("*.jsonl")], key=bin_start_year)
    print("No corpus manifest found; falling back to directory scan.")

missing = [b for b in bins if not (CORPUS_DIR / f"{b}.jsonl").exists()]
if missing:
    raise FileNotFoundError(f"Manifest lists missing bin files: {missing}")

manifest_counts = corpus_manifest.get("final_bin_counts", {}) if corpus_manifest else {}
manifest_hashes = corpus_manifest.get("bin_content_hashes", {}) if corpus_manifest else {}

count_mismatches = {}
hash_missing = []
for b in bins:
    if b in manifest_counts:
        with open(CORPUS_DIR / f"{b}.jsonl", "r", encoding="utf-8") as f:
            n_lines = sum(1 for _ in f)
        if n_lines != manifest_counts[b]:
            count_mismatches[b] = {"manifest": manifest_counts[b], "actual": n_lines}

    if corpus_manifest is not None and b not in manifest_hashes:
        hash_missing.append(b)

if count_mismatches:
    raise ValueError(f"Manifest/bin count mismatches: {count_mismatches}")

if hash_missing:
    raise ValueError(f"Manifest missing bin_content_hashes for bins: {hash_missing}")


# ── MODEL: load tokenizer and define layers ───────────────────────────────
log("Loading tokenizer/model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=True)
model = AutoModel.from_pretrained(
    MODEL_PATH,
    attn_implementation="eager"
).to(DEVICE)
log(f"Loaded SciBERT on {DEVICE}")

log(f"Layer modes selected: {LAYER_MODES}")

# ── RUN: executable ───────────────────────────────────────────────────────
for layer_mode in LAYER_MODES:
    EMB_DIR = EMB_ROOT / layer_mode
    EMB_DIR.mkdir(parents=True, exist_ok=True)

    for bin_label in tqdm(bins, desc=f"{layer_mode}"):
        in_path = CORPUS_DIR / f"{bin_label}.jsonl"
        out_pt = EMB_DIR / f"{bin_label}.pt"
        out_meta = EMB_DIR / f"{bin_label}.meta.json"

        expected_rows = manifest_counts.get(bin_label)
        expected_hash = manifest_hashes.get(bin_label)

        cache_ok = out_pt.exists() and out_meta.exists()
        if cache_ok:
            with open(out_meta, "r", encoding="utf-8") as f:
                meta = json.load(f)

            cached_rows = meta.get("rows")
            cached_hash = meta.get("corpus_bin_sha256")

            if expected_rows is not None and cached_rows != expected_rows:
                print(
                    f"Recomputing {layer_mode}/{bin_label}: "
                    f"cached rows={cached_rows}, expected rows={expected_rows}"
                )
                cache_ok = False

            if expected_hash is not None and cached_hash != expected_hash:
                print(
                    f"Recomputing {layer_mode}/{bin_label}: "
                    f"cached hash={cached_hash}, expected hash={expected_hash}"
                )
                cache_ok = False

        if cache_ok:
            continue

        recs = load_jsonl_records(in_path)
        X, n_failed = embed_mentions_centered(recs, tokenizer, model, layer_mode=layer_mode)
        torch.save(X, out_pt)

        meta = {
            "bin": bin_label,
            "model": MODEL_PATH,
            "representation": "mention_in_context_centered_window",
            "pool": "mention_mean",
            "layer_mode": layer_mode,
            "max_length": MAX_LENGTH,
            "batch_size": 8 if layer_mode == "weighted" else 16,
            "left_chars": LEFT_CHARS,
            "right_chars": RIGHT_CHARS,
            "device": str(DEVICE),
            "rows": int(X.shape[0]),
            "dim": int(X.shape[1]),
            "failed_span_count": int(n_failed),
            "failed_span_frac": float(n_failed / max(1, len(recs))),
            "corpus_bin_sha256": expected_hash,

        }
        with open(out_meta, "w", encoding="utf-8") as f:
            json.dump(meta, f, indent=2)

        log(f"{bin_label}: {X.shape} failed_span={n_failed} ({n_failed/len(recs):.2%})")

log(f"Done -> {EMB_ROOT}")
